# JSD Evaluation — ArtificialSocieties

Computes Jensen-Shannon Divergence between LLM-simulated survey responses and GSS ground truth distributions.  
Each output CSV from `query_gss_comprehensive.py` (one per model × persona file) is loaded and evaluated separately,  
then results are aggregated into a summary table.

**CSV schema expected** (from `query_gss_comprehensive.py` output):  
`timestamp, model, persona_file, persona_id, variable, question_short, run, answer, prompt_tokens, completion_tokens, total_tokens, error, raw_response`

## 0. Configuration — edit these paths

In [ ]:
from pathlib import Path

# ── Root of your ArtificialSocieties repo ─────────────────────────────────────
PROJECT_DIR = Path("/home/sant6886/ArtSoc/ArtificialSocieties")

# ── Synthetic data directory (where your per-model CSVs live) ─────────────────
# The script writes to: synthetic_data/year_{year}/{model}_{persona_file}.csv
YEAR = 2024
SYNTHETIC_DIR = PROJECT_DIR / "synthetic_data" / f"year_{YEAR}"

# ── GSS ground truth ──────────────────────────────────────────────────────────
# Path to actual GSS microdata CSV. Must contain one column per GSS variable
# with the same variable names used in GSS_QUESTIONS_COMPREHENSIVE.
# If you don't have this yet, set to None and only internal consistency
# metrics (homogenisation, demographic sensitivity) will be computed.
GSS_GROUND_TRUTH_CSV = PROJECT_DIR / "data" / "gss2024_actual.csv"  # adjust if needed

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = PROJECT_DIR / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Synthetic data dir : {SYNTHETIC_DIR}")
print(f"GSS ground truth   : {GSS_GROUND_TRUTH_CSV}")
print(f"Output dir         : {OUTPUT_DIR}")

## 1. Imports & core metric functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency, entropy
from scipy.spatial.distance import jensenshannon
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

# ─── Core metric helpers ──────────────────────────────────────────────────────

def normalise(series: pd.Series, options: list) -> np.ndarray:
    """Return a probability vector over `options`, handling missing categories."""
    counts = series.value_counts().reindex(options, fill_value=0)
    total = counts.sum()
    return (counts / total).values if total > 0 else np.ones(len(options)) / len(options)


def compute_jsd(p: np.ndarray, q: np.ndarray) -> float:
    """
    Jensen-Shannon Divergence (base-2 log, range 0-1).
    scipy.spatial.distance.jensenshannon returns the *square root*,
    so we square it to get the proper divergence measure.
    """
    js_dist = jensenshannon(p, q, base=2)
    return float(js_dist ** 2)  # JSD in [0, 1]


def compute_tvd(p: np.ndarray, q: np.ndarray) -> float:
    """Total Variation Distance: max per-category probability difference."""
    return float(0.5 * np.sum(np.abs(p - q)))


def homogenisation_score(llm_series: pd.Series, gss_series: pd.Series, options: list) -> dict:
    """Compare Shannon entropy of LLM vs GSS distributions.
    ratio < 1 => LLM is more homogeneous (less diverse) than real respondents.
    """
    p = normalise(llm_series, options)
    q = normalise(gss_series, options)
    H_llm = float(entropy(p + 1e-12, base=2))
    H_gss = float(entropy(q + 1e-12, base=2))
    return {
        "entropy_llm": H_llm,
        "entropy_gss": H_gss,
        "homogenisation_ratio": H_llm / H_gss if H_gss > 0 else np.nan,
    }


def demographic_sensitivity(df: pd.DataFrame, variable: str, demo_col: str) -> dict:
    """Chi-squared test: do LLM responses vary across demographic groups?"""
    subset = df[df["variable"] == variable].dropna(subset=["answer", demo_col])
    if subset[demo_col].nunique() < 2 or len(subset) < 10:
        return {"chi2": np.nan, "p_value": np.nan, "sensitive": np.nan}
    ct = pd.crosstab(subset[demo_col], subset["answer"])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return {"chi2": np.nan, "p_value": np.nan, "sensitive": np.nan}
    chi2, p, *_ = chi2_contingency(ct)
    return {"chi2": float(chi2), "p_value": float(p), "sensitive": p < 0.05}


def per_category_jsd_contribution(p: np.ndarray, q: np.ndarray) -> np.ndarray:
    """Per-category contribution to JSD (sums to JSD)."""
    m = 0.5 * (p + q)
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_p = np.where(p > 0, p * np.log2(p / m), 0.0)
        kl_q = np.where(q > 0, q * np.log2(q / m), 0.0)
    return np.maximum(0, 0.5 * kl_p + 0.5 * kl_q)


print("Core metric functions loaded.")

## 2. Load all synthetic CSV files

Each file is named `{model}_{persona_file}.csv`. We parse the filename to recover
`model` and `persona_file` — but we also trust the columns inside the CSV which
contain the same information explicitly.

In [ ]:
csv_files = sorted(SYNTHETIC_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s) in {SYNTHETIC_DIR}:\n")
for f in csv_files:
    print(f"  {f.name}")

if not csv_files:
    raise FileNotFoundError(
        f"No CSVs found. Check that SYNTHETIC_DIR is correct: {SYNTHETIC_DIR}"
    )

In [ ]:
dfs = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    df["source_file"] = f.name
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)

# Drop failed rows (no answer)
raw = raw[raw["answer"].notna() & (raw["error"].fillna("") == "")].copy()

# Normalise answer to string for consistent grouping
raw["answer"] = raw["answer"].astype(str).str.strip()

print(f"Total valid rows loaded: {len(raw):,}")
print(f"Models            : {sorted(raw['model'].unique())}")
print(f"Persona files     : {sorted(raw['persona_file'].unique())}")
print(f"Variables (q's)   : {raw['variable'].nunique()}")
raw.head(2)

## 3. Load GSS ground truth

If you don't have a ground-truth file yet, this cell gracefully skips JSD computation
and only runs the internal consistency metrics in Section 5.

In [ ]:
HAS_GROUND_TRUTH = GSS_GROUND_TRUTH_CSV is not None and Path(GSS_GROUND_TRUTH_CSV).exists()

if HAS_GROUND_TRUTH:
    gss_actual = pd.read_csv(GSS_GROUND_TRUTH_CSV, low_memory=False)
    # Normalise string columns
    for col in gss_actual.select_dtypes(include="object").columns:
        gss_actual[col] = gss_actual[col].astype(str).str.strip()
    print(f"GSS ground truth loaded: {len(gss_actual):,} rows, columns: {list(gss_actual.columns[:10])}...")
else:
    gss_actual = None
    print("⚠  No GSS ground truth found. JSD vs. actual GSS will be skipped.")
    print("   Set GSS_GROUND_TRUTH_CSV in cell 0 to enable it.")

## 4. Infer answer option sets per variable

We reconstruct the option set from what the LLMs actually responded with.  
If you have `GSS_QUESTIONS_COMPREHENSIVE` available, import it directly instead.

In [ ]:
# Option A: import options directly from the repo (preferred)
import sys
sys.path.insert(0, str(PROJECT_DIR))

try:
    # Adjust the import path to wherever GSS_QUESTIONS_COMPREHENSIVE is defined
    from generation.query_gss_comprehensive import GSS_QUESTIONS_COMPREHENSIVE
    variable_options = {var: list(meta["options"]) for var, meta in GSS_QUESTIONS_COMPREHENSIVE.items()}
    print(f"Loaded options for {len(variable_options)} variables from repo.")
except ImportError:
    # Option B: infer options from observed LLM answers + GSS actual data
    print("Could not import GSS_QUESTIONS_COMPREHENSIVE — inferring options from data.")
    variable_options = {}
    for var in raw["variable"].unique():
        opts = set(raw[raw["variable"] == var]["answer"].dropna().unique())
        if HAS_GROUND_TRUTH and var in gss_actual.columns:
            opts |= set(gss_actual[var].dropna().astype(str).str.strip().unique())
        variable_options[var] = sorted(opts)

print(f"Total variables with options: {len(variable_options)}")

## 5. Compute JSD — split by model × persona file × variable

In [ ]:
results = []

groups = raw.groupby(["model", "persona_file"])

for (model, persona_file), group_df in groups:
    print(f"\nProcessing: {model}  |  {persona_file}  ({len(group_df):,} rows)")

    for var in group_df["variable"].unique():
        options = variable_options.get(var)
        if not options:
            continue

        llm_answers = group_df[group_df["variable"] == var]["answer"].dropna()
        if len(llm_answers) < 5:
            continue

        p_llm = normalise(llm_answers, options)

        row = {
            "model": model,
            "persona_file": persona_file,
            "variable": var,
            "n_llm_responses": len(llm_answers),
        }

        # ── JSD vs GSS ground truth ──────────────────────────────────────────
        if HAS_GROUND_TRUTH and var in gss_actual.columns:
            gss_answers = gss_actual[var].dropna().astype(str).str.strip()
            p_gss = normalise(gss_answers, options)

            jsd_val  = compute_jsd(p_llm, p_gss)
            tvd_val  = compute_tvd(p_llm, p_gss)
            homo     = homogenisation_score(llm_answers, gss_answers, options)
            contrib  = per_category_jsd_contribution(p_llm, p_gss)

            row.update({
                "n_gss_responses": len(gss_answers),
                "jsd": jsd_val,
                "jsd_sqrt": float(np.sqrt(jsd_val)),   # metric distance
                "tvd": tvd_val,
                "entropy_llm": homo["entropy_llm"],
                "entropy_gss": homo["entropy_gss"],
                "homogenisation_ratio": homo["homogenisation_ratio"],
                # category with highest contribution to JSD
                "worst_category": options[int(np.argmax(contrib))],
                "worst_category_contrib": float(np.max(contrib)),
            })
        else:
            row.update({
                "n_gss_responses": np.nan,
                "jsd": np.nan, "jsd_sqrt": np.nan, "tvd": np.nan,
                "entropy_llm": float(entropy(p_llm + 1e-12, base=2)),
                "entropy_gss": np.nan,
                "homogenisation_ratio": np.nan,
                "worst_category": np.nan,
                "worst_category_contrib": np.nan,
            })

        results.append(row)

results_df = pd.DataFrame(results)
print(f"\nTotal evaluation rows: {len(results_df):,}")
results_df.head()

## 6. Summary table — mean JSD per model × persona file

In [ ]:
summary = (
    results_df
    .groupby(["model", "persona_file"])
    .agg(
        n_variables=("variable", "count"),
        mean_jsd=("jsd", "mean"),
        median_jsd=("jsd", "median"),
        mean_jsd_sqrt=("jsd_sqrt", "mean"),
        mean_tvd=("tvd", "mean"),
        mean_homogenisation_ratio=("homogenisation_ratio", "mean"),
        pct_vars_jsd_gt_015=("jsd", lambda x: (x > 0.15).mean() * 100),  # % vars with large gap
    )
    .reset_index()
    .sort_values("mean_jsd")
)

summary.style.background_gradient(subset=["mean_jsd", "mean_tvd"], cmap="RdYlGn_r")

## 7. Worst variables per model (highest JSD)

In [ ]:
if HAS_GROUND_TRUTH:
    TOP_N = 10
    for (model, pfile), grp in results_df.groupby(["model", "persona_file"]):
        print(f"\n{'─'*60}")
        print(f"  {model}  |  {pfile}")
        print(f"{'─'*60}")
        top = grp.nlargest(TOP_N, "jsd")[["variable", "jsd", "tvd", "homogenisation_ratio", "worst_category"]]
        print(top.to_string(index=False))
else:
    print("Ground truth not available — showing variables by LLM response entropy (lower = more homogenised)")
    for (model, pfile), grp in results_df.groupby(["model", "persona_file"]):
        print(f"\n{model}  |  {pfile}")
        print(grp.nsmallest(10, "entropy_llm")[["variable", "entropy_llm"]].to_string(index=False))

## 8. Heatmap — JSD per model × variable

Shows which model struggles most on which survey question.

In [ ]:
if HAS_GROUND_TRUTH and results_df["jsd"].notna().any():
    pivot = results_df.pivot_table(
        index="variable",
        columns=["model", "persona_file"],
        values="jsd",
        aggfunc="mean",
    )
    # Flatten multi-level columns for readability
    pivot.columns = [" | ".join(c).strip() for c in pivot.columns]

    fig_h = max(8, len(pivot) * 0.35)
    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 2.5), fig_h))
    sns.heatmap(
        pivot,
        ax=ax,
        cmap="RdYlGn_r",
        vmin=0, vmax=0.5,
        annot=True, fmt=".3f",
        linewidths=0.4,
        cbar_kws={"label": "JSD (0 = identical, 1 = completely different)"},
    )
    ax.set_title("JSD by variable × model | persona file", fontsize=13, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("GSS variable")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "jsd_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUTPUT_DIR / 'jsd_heatmap.png'}")
else:
    print("Skipped — no ground truth or all JSD values are NaN.")

## 9. Homogenisation analysis

Ratio < 1 means the LLM produces less diverse responses than real GSS respondents.  
This is the 'response homogenisation' failure mode from Barrie & Cerina (2026).

In [ ]:
if HAS_GROUND_TRUTH and results_df["homogenisation_ratio"].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: distribution of homogenisation ratios per model
    for model, grp in results_df.groupby("model"):
        grp["homogenisation_ratio"].dropna().plot.kde(ax=axes[0], label=model.split("/")[-1])
    axes[0].axvline(1.0, color="black", linestyle="--", linewidth=1)
    axes[0].set_xlabel("Homogenisation ratio (LLM entropy / GSS entropy)")
    axes[0].set_title("Distribution of homogenisation ratios")
    axes[0].legend(fontsize=8)
    axes[0].annotate("← more homogeneous", xy=(0.05, 0.92), xycoords="axes fraction", fontsize=9, color="red")
    axes[0].annotate("more diverse →", xy=(0.62, 0.92), xycoords="axes fraction", fontsize=9, color="green")

    # Right: scatter JSD vs homogenisation ratio
    for model, grp in results_df.groupby("model"):
        axes[1].scatter(
            grp["homogenisation_ratio"], grp["jsd"],
            alpha=0.5, s=20, label=model.split("/")[-1]
        )
    axes[1].axvline(1.0, color="black", linestyle="--", linewidth=1)
    axes[1].set_xlabel("Homogenisation ratio")
    axes[1].set_ylabel("JSD")
    axes[1].set_title("JSD vs. homogenisation ratio")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "homogenisation.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    # If no ground truth: show raw LLM entropy per model
    fig, ax = plt.subplots(figsize=(10, 4))
    for model, grp in results_df.groupby("model"):
        grp["entropy_llm"].dropna().plot.kde(ax=ax, label=model.split("/")[-1])
    ax.set_xlabel("Shannon entropy of LLM response distribution (bits)")
    ax.set_title("LLM response diversity per model (no ground truth)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "entropy_no_gt.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Demographic sensitivity — do LLMs condition on persona?

Joins LLM responses back to persona demographics, then runs χ² tests.
A non-significant result (p > 0.05) means the LLM is ignoring the persona.

In [ ]:
# Load persona files to get demographic columns
persona_file_map = {
    "personas_demographics_political": PROJECT_DIR / "data" / "personas_demographics_political.csv",
    "personas_demographics"          : PROJECT_DIR / "data" / "personas_demographics.csv",
    "gss2024_personas"               : PROJECT_DIR / "data" / "gss2024_personas.csv",
}

# Identify demographic columns available across persona files
demo_cols_by_file = {}
for tag, path in persona_file_map.items():
    if path.exists():
        p = pd.read_csv(path, nrows=2)
        numeric_or_cat = [c for c in p.columns if c not in ("respondent_id", "persona")]
        demo_cols_by_file[tag] = numeric_or_cat
        print(f"  {tag}: {numeric_or_cat}")

print()
DEMO_COLS = ["age", "sex", "race", "educ", "partyid", "polviews"]  # adjust to what's actually in your files

In [ ]:
sensitivity_results = []

for (model, persona_file), group_df in raw.groupby(["model", "persona_file"]):
    # Load matching persona file
    persona_path = persona_file_map.get(persona_file)
    if persona_path is None or not persona_path.exists():
        continue

    personas = pd.read_csv(persona_path)
    merged = group_df.merge(personas, on="respondent_id", how="left", suffixes=("", "_persona"))

    available_demos = [d for d in DEMO_COLS if d in merged.columns]

    for var in merged["variable"].unique():
        for demo in available_demos:
            sens = demographic_sensitivity(merged, var, demo)
            sensitivity_results.append({
                "model": model,
                "persona_file": persona_file,
                "variable": var,
                "demographic": demo,
                **sens,
            })

if sensitivity_results:
    sens_df = pd.DataFrame(sensitivity_results)

    # % of variables where LLM IS sensitive to each demographic
    pct_sensitive = (
        sens_df.groupby(["model", "persona_file", "demographic"])["sensitive"]
        .mean() * 100
    ).reset_index(name="pct_sensitive")

    print("% of variables where LLM responses are significantly sensitive to each demographic:")
    print(pct_sensitive.to_string(index=False))
else:
    print("No persona files found for demographic sensitivity analysis.")
    sens_df = pd.DataFrame()

## 11. Naïve baseline comparison

A model that just samples from the GSS marginal distribution should achieve some base JSD.  
If the LLM doesn't beat this baseline, it's not adding demographic conditioning value.

In [ ]:
if HAS_GROUND_TRUTH:
    rng = np.random.default_rng(42)
    N_SIM = 5_000

    baseline_rows = []
    for var, options in variable_options.items():
        if var not in gss_actual.columns:
            continue
        gss_answers = gss_actual[var].dropna().astype(str).str.strip()
        if len(gss_answers) < 5:
            continue
        p_gss = normalise(gss_answers, options)
        # Simulate a model that just draws from the marginal
        simulated = pd.Series(rng.choice(options, size=N_SIM, p=p_gss))
        p_sim = normalise(simulated, options)
        baseline_rows.append({
            "variable": var,
            "baseline_jsd": compute_jsd(p_sim, p_gss),
        })

    baseline_df = pd.DataFrame(baseline_rows)

    # Merge into results
    results_with_baseline = results_df.merge(baseline_df, on="variable", how="left")
    results_with_baseline["jsd_vs_baseline"] = (
        results_with_baseline["jsd"] - results_with_baseline["baseline_jsd"]
    )

    summary_baseline = (
        results_with_baseline
        .groupby(["model", "persona_file"])[["jsd", "baseline_jsd", "jsd_vs_baseline"]]
        .mean()
        .reset_index()
    )
    print("Mean JSD vs. naïve baseline (negative = LLM is better than random sampling from GSS marginal):")
    print(summary_baseline.to_string(index=False))
else:
    print("Baseline comparison requires GSS ground truth.")

## 12. Save results

In [ ]:
results_df.to_csv(OUTPUT_DIR / "jsd_by_variable.csv", index=False)
summary.to_csv(OUTPUT_DIR / "jsd_summary.csv", index=False)

if not sens_df.empty:
    sens_df.to_csv(OUTPUT_DIR / "demographic_sensitivity.csv", index=False)

print("Saved:")
for f in OUTPUT_DIR.glob("*.csv"):
    print(f"  {f}")